# TARGE — Five-Group Ablation Sweep (Colab)

Runs the five ablation groups against a single **already-trained** TARGE checkpoint.
Pipeline: setup → install → login → paths → data → checkpoint → POPE prep → oracle → sweep → table.

| Group | route_mode      | use_qformer | Notes |
|-------|-----------------|-------------|-------|
| A     | full            | False       | upper bound: all N tokens, no compression |
| B     | random_topk     | False       | lower bound: random k tokens |
| C     | oracle          | False       | indices precomputed from LLM cross-attention |
| D     | selector        | True        | trained behavior (status quo) |
| E     | selector        | False       | selector-only |

**Note:** Group F (`route_mode=full`, `use_qformer=True`) was removed. With `route_mode=full`, the projector's `dropped` tensor is empty, so the Q-Former operates on zero tokens and the connector output ends up nearly identical to Group A (cos vs A ≈ 0.9996). It's not a meaningful ablation as currently coded.

**Eval set:** POPE `popular` (COCO val2014 images) — disjoint from the LLaVA-CC-SBU pretrain shard the model was trained on, so the cos / IoU / generation pass is leak-free without retraining.


## 1. Setup: GPU check + mount Drive


In [ ]:
print("== GPU ==")
!nvidia-smi -L || echo "(no GPU detected)"
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || true

print("\n== Drive ==")
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")


## 2. Filepaths + install

Single source of truth for every path used in this notebook.

- **Drive** = `/content/drive/MyDrive/...` — persistent, survives session teardown, but slow (FUSE-backed). Use only for the authoritative source code (read) and durable outputs we want to keep (write).
- **Local** = `/content/...` — the VM's NVMe — fast, lost when the session ends. Everything compute-related (source mirror, dataset, HF cache, intermediates) lives here.

This cell also rsync-mirrors the source from Drive to local SSD and runs `pip install -e .` from the mirror, because `pip install -e` writes a lot of small files (egg-info, .pth) and that's painful on Drive.


In [ ]:
import os, time
from pathlib import Path

# ─── Tunable knobs ────────────────────────────────────────────────────────────
RUN_ID      = "targe-smollm2-5k"   # which trained run to evaluate
POPE_SUBSET = "popular"            # POPE category for held-out eval
N_HELDOUT   = 500                  # heldout entries for cos/IoU/gen pass

# ─── Drive (persistent — source of truth + durable outputs) ──────────────────
DRIVE_ROOT   = Path("/content/drive/MyDrive/targe-prismatic-vlms")  # repo source (read)
CKPT_DIR     = DRIVE_ROOT / "runs" / RUN_ID                          # trained checkpoint (read)
RESULTS_JSON = CKPT_DIR / "ablation_results.json"                    # final eval output (write)

# ─── Local (VM SSD — compute scratch) ────────────────────────────────────────
LOCAL_ROOT   = Path("/content")
REPO_DIR     = LOCAL_ROOT / "repo"                                   # mirror of DRIVE_ROOT
TAR_PATH     = LOCAL_ROOT / "train_subset_5k.tar"                    # input tarball (training)
DATA_DIR     = LOCAL_ROOT / "data" / "download" / "llava-laion-cc-sbu-558k"
CHAT_JSON    = DATA_DIR / "chat.json"
POPE_DIR     = LOCAL_ROOT / "pope"                                   # POPE root (--image_root for eval)
HELDOUT_JSON = POPE_DIR / "chat_heldout.json"
POPE_JSON    = POPE_DIR / f"pope_{POPE_SUBSET}.json"
ORACLE_PT    = LOCAL_ROOT / "oracle_indices.pt"                      # intermediate; not persisted
HF_CACHE     = LOCAL_ROOT / "hf_cache"                               # MUST be local — HF uses symlinks, Drive doesn't support them

# ─── Rescue CWD (in case a prior Drive remount left us in a vanished dir) ──
os.chdir(LOCAL_ROOT)

# ─── Create dirs + wire HF env vars ──────────────────────────────────────────
for d in (DATA_DIR.parent, POPE_DIR, HF_CACHE, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"]      = str(HF_CACHE)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE / "hub")

# ─── Mirror source Drive → local SSD (incremental rsync) ───────────────────
# Excludes are critical: hf_cache contains HF symlinks Drive's FUSE can't stat
# (`Operation not supported (95)`), and the egg-info / pyc dirs are build
# artifacts that get regenerated on `pip install -e`.
t0 = time.time()
REPO_DIR.mkdir(parents=True, exist_ok=True)
!rsync -a \
    --exclude='runs' \
    --exclude='data' \
    --exclude='hf_cache' \
    --exclude='*.tar' \
    --exclude='__pycache__' \
    --exclude='*.pyc' \
    --exclude='.git' \
    --exclude='*.egg-info' \
    --exclude='build' \
    --exclude='dist' \
    --exclude='.ipynb_checkpoints' \
    "{DRIVE_ROOT}/" "{REPO_DIR}/"
print(f"[mirror] {time.time()-t0:.1f}s")

# Sanity-check: pyproject.toml must be present before pip install.
PYPROJECT = REPO_DIR / "pyproject.toml"
assert PYPROJECT.exists(), (
    f"rsync finished but {PYPROJECT} is missing — likely a Drive FUSE crash mid-copy.\n"
    "Fix: re-mount Drive ( drive.mount('/content/drive', force_remount=True) ),\n"
    "     then `!rm -rf /content/repo` and re-run this cell."
)

# ─── pip install -e from local SSD (cached after first run) ─────────────────
%cd {REPO_DIR}
INSTALLED_MARKER = REPO_DIR / ".pip_installed"
if not INSTALLED_MARKER.exists():
    t0 = time.time()
    !pip install -e . --no-build-isolation --quiet
    !pip install -q fvcore || echo "[install] fvcore failed — FLOPS column will be null"
    INSTALLED_MARKER.touch()
    print(f"[install] {time.time()-t0:.1f}s")
else:
    print("[install] cached  (remove /content/repo/.pip_installed to force reinstall)")

# ─── Echo canonical layout ───────────────────────────────────────────────────
print("\n" + "─" * 60)
print(f"Drive source : {DRIVE_ROOT}")
print(f"Drive ckpt   : {CKPT_DIR}  ({'exists' if CKPT_DIR.exists() else 'MISSING'})")
print(f"Drive results: {RESULTS_JSON}")
print(f"Local repo   : {REPO_DIR}")
print(f"Local tar    : {TAR_PATH}   ({'exists' if TAR_PATH.exists() else 'MISSING'})")
print(f"Local data   : {DATA_DIR}   ({'extracted' if DATA_DIR.exists() else 'not yet'})")
print(f"Local POPE   : {POPE_DIR}   ({'prepared' if HELDOUT_JSON.exists() else 'not yet'})")
print(f"Local oracle : {ORACLE_PT}")
print(f"HF cache     : {HF_CACHE}")


## 3. Logins (Hugging Face, W&B)


In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login, whoami

# Hugging Face
hf_token = userdata.get('hf_token')
login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token
who = whoami()
print(f"[hf] logged in as: {who.get('name')}  (orgs: {[o['name'] for o in who.get('orgs', [])]})")


## 4. Untar dataset into the layout `llava-v15` expects

Idempotent: skips if `chat.json` is already present. Extracts to `/content/data/download/llava-laion-cc-sbu-558k/` and renames `chat_subset_5k.json` → `chat.json` so the default `LLaVa_V15_Config` paths resolve.


In [ ]:
import time, tarfile

OLD_DIR = Path("/content/train_subset_5k")

assert TAR_PATH.exists() or OLD_DIR.exists() or CHAT_JSON.exists(), (
    f"Need either {TAR_PATH}, {OLD_DIR}, or extracted {CHAT_JSON}."
)

if CHAT_JSON.exists():
    n_files = sum(1 for _ in DATA_DIR.rglob("*") if _.is_file())
    print(f"[skip] {DATA_DIR} already populated ({n_files:,} files)")
else:
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)

    if OLD_DIR.exists() and any(OLD_DIR.iterdir()):
        print(f"[move] {OLD_DIR} -> {DATA_DIR}")
        if DATA_DIR.exists():
            DATA_DIR.rmdir()
        OLD_DIR.rename(DATA_DIR)
    else:
        with tarfile.open(TAR_PATH) as tf:
            tops = set()
            for i, m in enumerate(tf):
                tops.add(m.name.split("/", 1)[0])
                if i > 20:
                    break
        has_wrapper = len(tops) == 1 and next(iter(tops)) not in {"", "."}
        strip_flag = "--strip-components 1" if has_wrapper else ""
        print(f"[tar] top-level sample : {sorted(tops)[:5]}{'...' if len(tops) > 5 else ''}")
        print(f"[tar] strip wrapper dir? {has_wrapper}")

        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print(f"[tar] extracting -> {DATA_DIR}  ({TAR_PATH.stat().st_size / 1e9:.2f} GB)")
        t0 = time.time()
        !tar -xf "{TAR_PATH}" {strip_flag} -C "{DATA_DIR}"
        print(f"[tar] done in {time.time()-t0:.1f}s")

    src_json = DATA_DIR / "chat_subset_5k.json"
    if src_json.exists() and not CHAT_JSON.exists():
        src_json.rename(CHAT_JSON)
        print(f"[rename] {src_json.name} -> {CHAT_JSON.name}")

if not CHAT_JSON.exists():
    hits = list(Path("/content").rglob("chat.json")) + list(Path("/content").rglob("chat_subset_5k.json"))
    raise FileNotFoundError(f"Expected {CHAT_JSON} but missing. Found candidates: {hits}")

print(f"\n[ok] {CHAT_JSON} ({CHAT_JSON.stat().st_size / 1e6:.2f} MB)")


## 5. Train the connector (align stage)

Runs `scripts/pretrain.py` to train the selector + projector against the LLaVA-CC-SBU subset. Idempotent — skips if `latest-checkpoint.pt` already exists under `CKPT_DIR/checkpoints/`.

- Launched from `/content/` so prismatic's default relative `data/…` paths resolve to `LOCAL_DATA_DIR`.
- Checkpoints + `train.log` land on Drive via `--run_root_dir` (so they survive the VM teardown).
- Tunable knobs (`MODEL_TYPE`, `BSZ`, `LR`, `MAX_STEPS`, …) live at the top of the cell.
- `num_workers` is auto-set inside the strategy (`min(8, cpu_count)`); override with `--num_workers N` if needed.
- The first-backward gradient audit will print `||grad||` for every projector param so you can confirm the selector router actually receives gradient.


In [ ]:
import os, time
os.environ["PYTHONUNBUFFERED"] = "1"

# ─── Training hyperparameters (align stage) ──────────────────────────────────
MODEL_TYPE    = "targe-smollm2-360m-clipb-224px"
STAGE         = "align"
BSZ           = 14                  # per-device == global (single GPU)
LR            = 1e-4
MAX_STEPS     = 500
DATASET_TYPE  = "llava-v15"
WANDB_ENTITY  = "nbusich-northeastern-university"
WANDB_PROJECT = "targe"

LATEST_CKPT = CKPT_DIR / "checkpoints" / "latest-checkpoint.pt"
TRAIN_LOG   = CKPT_DIR / "train.log"

if LATEST_CKPT.exists():
    print(f"[skip] checkpoint already exists: {LATEST_CKPT}")
    print(f"       (delete the file to force retrain)")
else:
    HF_TOKEN_VAL = os.environ.get("HF_TOKEN", "").strip()
    WB_KEY_VAL   = os.environ.get("WANDB_API_KEY", "").strip()
    TRACKERS     = "[jsonl,wandb]" if WB_KEY_VAL else "[jsonl]"
    TRAIN_LOG.parent.mkdir(parents=True, exist_ok=True)

    PRETRAIN = REPO_DIR / "scripts" / "pretrain.py"

    print(f"[train] run_id    : {RUN_ID}")
    print(f"[train] model     : {MODEL_TYPE}")
    print(f"[train] stage     : {STAGE}")
    print(f"[train] bsz/lr    : {BSZ} / {LR}")
    print(f"[train] max_steps : {MAX_STEPS}")
    print(f"[train] data      : {DATA_DIR}")
    print(f"[train] ckpts to  : {CKPT_DIR}/checkpoints/  (Drive — durable)")
    print(f"[train] log to    : {TRAIN_LOG}")
    print(f"[train] wandb     : {'on (' + WANDB_PROJECT + ')' if WB_KEY_VAL else 'off (no WANDB_API_KEY)'}")
    print(f"[train] started   : {time.strftime('%Y-%m-%d %H:%M:%S')}\n")

    # Launch from /content so the LLaVa_V15_Config default `data/…` resolves to {DATA_DIR}.
    %cd {LOCAL_ROOT}
    !set -o pipefail; HF_TOKEN="{HF_TOKEN_VAL}" HUGGING_FACE_HUB_TOKEN="{HF_TOKEN_VAL}" WANDB_API_KEY="{WB_KEY_VAL}" HF_HOME="{HF_CACHE}" \
      stdbuf -oL -eL torchrun --standalone --nnodes 1 --nproc-per-node 1 "{PRETRAIN}" \
        --model.type "{MODEL_TYPE}" \
        --stage "{STAGE}" \
        --model.align_per_device_batch_size {BSZ} \
        --model.align_global_batch_size {BSZ} \
        --model.align_learning_rate {LR} \
        --model.align_max_steps {MAX_STEPS} \
        --dataset.type "{DATASET_TYPE}" \
        --run_id "{RUN_ID}" \
        --run_root_dir "{CKPT_DIR.parent}" \
        --hf_token HF_TOKEN \
        --trackers '{TRACKERS}' \
        --wandb_entity "{WANDB_ENTITY}" \
        --wandb_project "{WANDB_PROJECT}" 2>&1 | tee -a "{TRAIN_LOG}"

    print(f"\n[train] finished  : {time.strftime('%Y-%m-%d %H:%M:%S')}")
    assert LATEST_CKPT.exists(), (
        f"Training process exited but {LATEST_CKPT} is missing — check {TRAIN_LOG} for errors."
    )


## 6. Locate the trained checkpoint

Lists all checkpoints under `CKPT_DIR` (on Drive). The ablation scripts load the model via `prismatic.load(CKPT_DIR)`, which picks up `config.json` + the latest checkpoint under `checkpoints/`.


In [ ]:
ckpts = sorted((CKPT_DIR / "checkpoints").glob("*.pt"))
print(f"[ckpt] run dir   : {CKPT_DIR}")
print(f"[ckpt] checkpoints found ({len(ckpts)}):")
for c in ckpts:
    sz = c.stat().st_size / 1e9
    print(f"   - {c.name:40s}  {sz:6.2f} GB")

assert ckpts, f"No checkpoints found in {CKPT_DIR / 'checkpoints'}. Train first (TARGE.ipynb section 7) or fix RUN_ID."
LATEST = ckpts[-1]
print(f"\n[ckpt] latest    : {LATEST.name}")


## 7. Prepare POPE held-out set

Downloads POPE `popular` via HuggingFace `lmms-lab/POPE`, materializes images to `POPE_DIR/images/`, and writes two JSON files:

- `pope_popular.json` (~3 000 entries) — passed as `--pope_json` for yes/no accuracy.
- `chat_heldout.json` (`N_HELDOUT` entries, seeded shuffle) — passed as `--heldout_json` for cos_vs_A / oracle IoU / generation.

POPE uses COCO val2014 images, which are **disjoint** from the LLaVA-CC-SBU-558K pretrain shard the model trained on — so the cos/IoU/gen pass is leak-free without retraining. Idempotent.

**Caveat:** the trained checkpoint is align-stage only (image-caption objective), not instruction-tuned for yes/no QA, so absolute POPE accuracy may be near chance. The signal is the **relative ordering** across the five ablation groups (does Oracle C beat Random B?).


In [ ]:
if HELDOUT_JSON.exists() and POPE_JSON.exists():
    import json
    with open(HELDOUT_JSON) as f:
        n_held = len(json.load(f))
    with open(POPE_JSON) as f:
        n_pope = len(json.load(f))
    print(f"[skip] POPE already prepared at {POPE_DIR}")
    print(f"       heldout : {n_held:,}")
    print(f"       pope    : {n_pope:,}")
else:
    print("[install] datasets (HF) — needed by scripts/eval/prepare_pope.py")
    !pip install -q datasets

    %cd /content/drive/MyDrive/targe-prismatic-vlms
    HF_TOKEN_VAL = os.environ.get("HF_TOKEN", "").strip()
    !HF_TOKEN="{HF_TOKEN_VAL}" HUGGING_FACE_HUB_TOKEN="{HF_TOKEN_VAL}" HF_HOME="/content/hf_cache" \
      python scripts/eval/prepare_pope.py \
        --out_dir "{POPE_DIR}" \
        --subset popular \
        --n_heldout 500 \
        --hf_token HF_TOKEN

    # Sanity-check both files were written.
    assert HELDOUT_JSON.exists(), f"POPE prep finished but {HELDOUT_JSON} is missing"
    assert POPE_JSON.exists(),    f"POPE prep finished but {POPE_JSON} is missing"


## 8. Precompute oracle indices

For each held-out example, run a forward with `route_mode="full"` + `use_qformer=False` so the LLM sees every projected visual token. Then average attention from response-token positions to visual-token positions across the early layers and take top-k. Saved as `ORACLE_PT` (local — intermediate, not persisted to Drive; cheap to recompute).


In [ ]:
# Drop any stale oracle file — IDs depend on the heldout source, so a session reset is safest.
ORACLE_PT.unlink(missing_ok=True)

%cd {REPO_DIR}
HF_TOKEN_VAL = os.environ.get("HF_TOKEN", "").strip()
!HF_TOKEN="{HF_TOKEN_VAL}" HUGGING_FACE_HUB_TOKEN="{HF_TOKEN_VAL}" HF_HOME="{HF_CACHE}" \
  python scripts/eval/precompute_oracle.py \
    --model_path "{CKPT_DIR}" \
    --heldout_json "{HELDOUT_JSON}" \
    --image_root "{POPE_DIR}" \
    --out_pt "{ORACLE_PT}" \
    --hf_token HF_TOKEN


## 9. Run the 5-group ablation sweep

For each POPE example, runs all five routing configurations against the same checkpoint, capturing generations, connector-output latents (for cosine similarity vs. Group A), Oracle IoU (groups D, E), and per-group hardware costs (FLOPS via `fvcore` if available, latency via CUDA events). POPE-style yes/no accuracy runs over the full POPE subset. Results are flushed atomically to `RESULTS_JSON` (on Drive — durable) after every example.

(Group F was removed — see the intro note. `GROUPS` in `scripts/eval/run_ablation.py` is now A, B, C, D, E only.)


In [ ]:
%cd {REPO_DIR}
HF_TOKEN_VAL = os.environ.get("HF_TOKEN", "").strip()
!HF_TOKEN="{HF_TOKEN_VAL}" HUGGING_FACE_HUB_TOKEN="{HF_TOKEN_VAL}" HF_HOME="{HF_CACHE}" \
  python scripts/eval/run_ablation.py \
    --model_path "{CKPT_DIR}" \
    --heldout_json "{HELDOUT_JSON}" \
    --image_root "{POPE_DIR}" \
    --oracle_pt "{ORACLE_PT}" \
    --out_json "{RESULTS_JSON}" \
    --pope_json "{POPE_JSON}" \
    --pope_image_root "{POPE_DIR}" \
    --hf_token HF_TOKEN


## 10. Results table


In [ ]:
import json
import pandas as pd

with open(RESULTS_JSON) as f:
    res = json.load(f)

rows = []
for name, s in res["summary"].items():
    hw = s.get("hardware") or {}
    pope = s.get("pope") or {}
    rows.append({
        "group":             name,
        "n":                 s.get("n_generations"),
        "cos_vs_A":          s.get("cos_vs_A_mean"),
        "iou_vs_oracle":     s.get("iou_vs_oracle_mean"),
        "pope_acc":          pope.get("accuracy"),
        "pope_yes_rate":     pope.get("yes_rate"),
        "flops":             hw.get("flops"),
        "latency_ms_median": hw.get("latency_ms_median"),
    })
df = pd.DataFrame(rows).set_index("group")
print(df.to_string(float_format=lambda v: f"{v:.4f}" if isinstance(v, float) else str(v)))


### 10b. Spot-check generations

Compare what each group says for a few held-out prompts.


In [ ]:
from IPython.display import display, Markdown

gens = res["generations"]
group_names = list(gens.keys())
# Index by example id within each group so we can align across groups.
by_id = {g: {item["id"]: item for item in gens[g]} for g in group_names}
common_ids = sorted(set.intersection(*[set(d.keys()) for d in by_id.values()]))
print(f"[gens] {len(common_ids)} examples present in every group")

for ex_id in common_ids[:3]:
    prompt = by_id[group_names[0]][ex_id]["prompt"]
    lines = [f"**id:** `{ex_id}`", f"**prompt:** {prompt}", ""]
    for g in group_names:
        lines.append(f"- **{g}:** {by_id[g][ex_id]['gen']}")
    display(Markdown("\n".join(lines)))
    display(Markdown("---"))
